# Week 6 — Spark Assignment
### Understanding Spark Architecture & Efficient DataFrame Operations

**Objective:** Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats.

**Dataset:** `data/source.csv` (sample e-commerce order/product dataset)

**Output:** Spark code (PySpark) + execution results + brief insights on performance and architecture.


## Setup — Start Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Week6-SparkAssignment") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


## Q1: Roles of Driver, Cluster Manager, and Executor

- **Driver**: Runs the `main()` program, creates the `SparkSession`, builds the logical execution plan (DAG), and schedules tasks across executors. It coordinates the entire application and collects final results.
- **Cluster Manager**: Allocates resources (CPU/memory) to the Spark application across the cluster. Examples: Standalone, YARN, Kubernetes, Mesos. It does not execute your code — only negotiates resources for the Driver and Executors.
- **Executor**: A worker process running on a cluster node. It executes the actual tasks assigned by the Driver, stores data in memory/disk (for caching/shuffles), and reports status/results back to the Driver.


## Q2: How Lazy Evaluation Improves Performance

Spark does **not** execute transformations (`filter`, `select`, `withColumn`, etc.) immediately — it only builds a logical plan (DAG). Execution happens only when an **action** (`show()`, `count()`, `write()`) is called.

**Why this helps performance:**
- Spark's **Catalyst Optimizer** can analyze the *entire* chain of transformations before running anything, and reorder/combine/eliminate redundant steps.
- Unnecessary computations and data reads are skipped — only what's needed for the final action is processed.
- It enables optimizations like **predicate pushdown** and **column pruning** across the whole pipeline, not just one step at a time.


In [ ]:
# Demonstration: nothing executes here (all lazy)
df_demo = spark.read.csv("data/source.csv", header=True, inferSchema=True)
df_lazy = df_demo.filter(col("category") == "Electronics").select("product_id", "price")
print("Transformations defined, but NOT yet executed.")

# Execution only happens now, on the action:
df_lazy.show(5)


## Q3: Read CSV with Header and Schema Inference

In [ ]:
df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)


## Q4: CSV vs Parquet — Storage & Performance

| | CSV | Parquet |
|---|---|---|
| Layout | Row-based | Columnar |
| Schema | Not embedded (inferred) | Embedded in file |
| Compression | Poor | Efficient (per-column) |
| Column pruning | Not possible | Possible |

**Why it matters:** In columnar storage, if a query needs only a few columns, Spark reads *only those columns* from disk. In row-based CSV, Spark must scan entire rows even when most columns are irrelevant to the query. This makes Parquet significantly faster and cheaper for analytical workloads on wide datasets.


In [ ]:
# Write the dataset to Parquet to compare
df.write.mode("overwrite").parquet("output/source_parquet")

# Read back and compare column-selective query cost (conceptually)
df_parquet = spark.read.parquet("output/source_parquet")
df_parquet.select("product_id", "price").show(5)


## Q5: Select `product_id` and `price` Where `category == 'Electronics'`

In [ ]:
result_q5 = df.filter(col("category") == "Electronics").select("product_id", "price")
result_q5.show()


## Q6: Rename `old_name` -> `new_name` and Cast `price` to Double

In [ ]:
df_q6 = df.withColumnRenamed("old_name", "new_name") \
          .withColumn("price", col("price").cast("double"))

df_q6.printSchema()
df_q6.select("product_id", "new_name", "price").show(5)


## Q7: Lineage Graph (DAG) and Fault Tolerance

Spark tracks every transformation applied to a DataFrame/RDD as a **lineage graph** -- a record of exactly how each dataset was derived from its source. If a worker node fails and loses partitions of data, Spark does **not** need to restart the whole job or rely on data replication for fault tolerance. Instead, it uses the lineage graph to identify which transformations produced the lost partitions and **recomputes only those partitions** from the original source. This makes RDDs/DataFrames resilient without the storage overhead of full replication.


## Q8: Filter `status == 'Completed'` AND `amount > 1000`

In [ ]:
df_orders = df  # using source dataset as df_orders
result_q8 = df_orders.filter((col("status") == "Completed") & (col("amount") > 1000))
result_q8.show()


## Q9: Predicate Pushdown in Parquet

Predicate Pushdown means filter conditions (`.filter()` / `WHERE`) are pushed down to the **file-reading layer** instead of being applied after loading all data into memory. Parquet stores column-level statistics (min/max, null counts) per row group, so Spark can **skip entire row groups** that cannot possibly satisfy the filter -- without even reading them from disk. This drastically reduces the amount of data read from disk and loaded into memory, especially on large datasets.


In [ ]:
# Predicate pushdown happens automatically when filtering a Parquet source
df_parquet.filter(col("amount") > 1000).explain(True)  # check the physical plan for PushedFilters


## Q10: Add Column `final_price` = `base_price` * 1.18 (18% tax)

In [ ]:
df_q10 = df.withColumn("final_price", col("base_price") * 1.18)
df_q10.select("product_id", "base_price", "final_price").show(5)


## Q11: Transformations vs Actions

- **Transformations** (lazy -- return a new DataFrame/RDD, don't execute immediately):
  - `filter()` -- e.g., `df.filter(col("price") > 500)`
  - `select()` -- e.g., `df.select("product_id", "price")`

- **Actions** (trigger execution of the full DAG, return results/write data):
  - `show()` -- e.g., `df.show()`
  - `count()` -- e.g., `df.count()`


## Q12: Read Parquet -> Filter Nulls (`user_id`) -> Write as CSV

In [ ]:
df_input = spark.read.parquet("output/source_parquet")  # simulating "path/to/input"
df_clean = df_input.filter(col("user_id").isNotNull())
df_clean.write.mode("overwrite").option("header", True).csv("output/cleaned_csv")  # "path/to/output"

df_clean.show()


## Q13: Client Mode vs Cluster Mode

- **Client Mode**: The Driver runs on the machine that submitted the application (e.g., a local laptop or edge node), *outside* the cluster. Good for interactive work and debugging, but if the client machine disconnects or crashes, the whole job dies.
- **Cluster Mode**: The Driver runs *inside* the cluster itself, managed by the Cluster Manager, on one of the worker nodes. This is preferred for production jobs since the application doesn't depend on the submitting machine staying connected.


## Q14: Filter `region == 'North'` OR `priority == 'High'`

In [ ]:
result_q14 = df.filter((col("region") == "North") | (col("priority") == "High"))
result_q14.show()


## Q15: Why `.show(5)` Instead of `.collect()` on a Multi-Terabyte Dataset

`.collect()` pulls **every row** of the DataFrame from all executors back into the Driver's memory. On a multi-terabyte dataset, this will overwhelm the Driver (which typically has far less memory than the whole cluster combined) and crash the application with an `OutOfMemoryError`.

`.show(5)` (or `.show()`) only computes and displays a **small sample** of rows (5, or 20 by default) without ever bringing the full dataset into Driver memory -- making it a safe way to explore large datasets.


In [ ]:
# Safe exploration on a large dataset
df.show(5)          # SAFE -- only fetches a small sample
# df.collect()       # UNSAFE on large datasets -- avoid this in real pipelines


## End-to-End Pipeline (Read -> Transform -> Filter -> Write)

Bringing together schema handling, filtering, transformations, and optimized output format into one pipeline.


In [ ]:
# 1. Read
pipeline_df = spark.read.csv("data/source.csv", header=True, inferSchema=True)

# 2. Transform (rename, cast, add derived column)
pipeline_df = pipeline_df.withColumnRenamed("old_name", "product_name") \
                          .withColumn("price", col("price").cast("double")) \
                          .withColumn("final_price", col("base_price") * 1.18)

# 3. Filter (drop nulls, business condition)
pipeline_df = pipeline_df.filter(col("user_id").isNotNull()) \
                          .filter(col("status") == "Completed")

# 4. Write (optimized columnar format)
pipeline_df.write.mode("overwrite").parquet("output/final_pipeline_output")

print("Pipeline complete. Row count:", pipeline_df.count())
pipeline_df.show(5)


## Insights -- Performance & Architecture

- **Lazy evaluation + DAG optimization** meant no computation ran until `.show()`/`.count()`/`.write()` were called, letting Catalyst optimize the whole chain at once.
- **Parquet vs CSV**: writing to Parquet enabled column pruning and predicate pushdown (confirmed via `.explain()` showing `PushedFilters`), which would meaningfully cut I/O on a large dataset.
- **Driver vs Executor separation** means the Driver only ever needs to hold the DAG and final aggregated/sample results -- never the full dataset -- which is why `.collect()` is dangerous at scale and `.show()`/writing to disk is the safe pattern.
- **Null handling** (`user_id`) and **type casting** (`price` to double) were applied before aggregation/filtering to ensure data quality downstream.
